In [2]:

import pandas as pd
import numpy as np
import os
import joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Load environment variables and establish a connection to the database
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# Retrieve training data from the database
X_train = pd.read_sql('SELECT * FROM "X_train"', engine)
y_train = pd.read_sql('SELECT * FROM "y_train"', engine).values.ravel()

# Define numerical ranks for categorical features that have a natural order
complex_mapping = {
    'exter_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'exter_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'heating_qc': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'kitchen_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'fireplace_qu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'pool_qc': {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'None': 0},
    'lot_shape': {'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1},
    'land_slope': {'Gtl': 3, 'Mod': 2, 'Sev': 1},
    'bsmt_exposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
    'garage_finish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0},
    'utilities': {'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1},
    'functional': {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
}

def ordinal_encode(df, mapping):
    #Apply ordinal encoding to categorical columns based on the provided mapping dictionary.
    df = df.copy()
    for col, m in mapping.items():
        if col in df.columns:
            df[col] = df[col].map(m).fillna(0).astype(int)
    return df

# Transform the training data using our ordinal encoding function
X_train_encoded = ordinal_encode(X_train, complex_mapping)

# Identify numerical and categorical features for separate processing
numeric_features = X_train_encoded.select_dtypes(include='number').columns.tolist()
categorical_features = X_train_encoded.select_dtypes(include='object').columns.tolist()

# Define a unified preprocessing pipeline for both feature types
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])

# Initialize a dictionary of regression models for benchmarking
models = {
    'XGBoost': XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(n_estimators=1000, learning_rate=0.05, depth=6, random_state=42, verbose=0),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Ridge': Ridge(alpha=1.0)
}

# Store cross-validation results to compare model performances later
results = []

for name, model in models.items():
    # Build a temporary pipeline for each candidate model
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Evaluate performance using 5-fold cross-validation on the training set
    cv_scores = cross_val_score(pipeline, X_train_encoded, y_train, cv=5, scoring='r2')
    
    results.append({
        'Model': name,
        'R2_Mean': cv_scores.mean(),
        'R2_Std': cv_scores.std()
    })
    print(f"Evaluated {name}... Mean R2: {cv_scores.mean():.4f}")

# Display the final comparison table sorted by the best mean R2 score
results_df = pd.DataFrame(results).sort_values(by='R2_Mean', ascending=False)
print(results_df)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17196\4154341703.py:60: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train_encoded.select_dtypes(include='object').columns.tolist()


Evaluated XGBoost... Mean R2: 0.8906


c:\Users\ASUS\Documents\A-Machine Learning projects\ames_housing_project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\Documents\A-Machine Learning projects\ames_housing_project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\Documents\A-Machine Learning projects\ames_housing_project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\Documents\A-Machine Learning projects\ames_housing_project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\ASUS\Docume

Evaluated LightGBM... Mean R2: 0.9037
Evaluated CatBoost... Mean R2: 0.9101
Evaluated RandomForest... Mean R2: 0.8787
Evaluated Ridge... Mean R2: 0.8965
          Model   R2_Mean    R2_Std
2      CatBoost  0.910053  0.013244
1      LightGBM  0.903678  0.010993
4         Ridge  0.896514  0.014712
0       XGBoost  0.890591  0.013774
3  RandomForest  0.878661  0.013097
